Instalar Go en Colab

In [16]:
!apt-get update
!apt-get install golang-go

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Fetched 3,917 B in 1s (4,850 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
golang-go is already the newest version (2:1.22~2build1).
0 upgraded, 0 newly installed, 0 to remove and 26 not upgraded.


In [11]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Implementación K-Means Secuencial (Go)

Este directorio contiene la implementación tradicional del algoritmo de clustering K-Means para el proyecto Quokka.

## Características de la Implementación
- **Lectura Estructurada:** Ingesta del dataset pivotado `dataset_kmeans_go.csv` donde cada observación es un vector de 5 dimensiones (PM10, PM2.5, SO₂, NO₂, CO).
- **Distancia Euclidiana:** Cálculo matemático de distancias entre las observaciones y los centroides en un espacio de 5 dimensiones.
- **Asignación y Actualización:** Ciclo iterativo clásico que reasigna clusters y recalcula los centroides hasta alcanzar la convergencia (0 cambios entre iteraciones) o el límite máximo de iteraciones.
- **Métricas Base:** Incorpora el paquete `time` para medir el tiempo exacto de ejecución, estableciendo el $T_{Secuencial}$.

In [19]:
%%writefile main_secuencial.go
package main

import (
	"encoding/csv"
	"fmt"
	"math"
	"math/rand"
	"os"
	"strconv"
	"time"
)

// Observacion representa una fila del dataset con sus 5 contaminantes.
type Observacion struct {
	Features []float64 // Vector multivariable: [PM10, PM2.5, SO2, NO2, CO]
	Cluster  int       // Índice del centroide al que pertenece
}

// calcularDistancia aplica la fórmula de distancia euclidiana entre dos vectores.
func calcularDistancia(a, b []float64) float64 {
	sum := 0.0
	for i := range a {
		diff := a[i] - b[i]
		sum += diff * diff
	}
	return math.Sqrt(sum)
}

// kMeansSecuencial ejecuta el algoritmo de agrupamiento en un solo hilo.
func kMeansSecuencial(datos []Observacion, k int, maxIter int) [][]float64 {
	numFeatures := len(datos[0].Features)
	centroides := make([][]float64, k)

	// 1. Inicialización: Seleccionar 'k' centroides iniciales al azar
	rand.Seed(time.Now().UnixNano())
	for i := 0; i < k; i++ {
		idx := rand.Intn(len(datos))
		centroides[i] = make([]float64, numFeatures)
		copy(centroides[i], datos[idx].Features)
	}

	// 2. Bucle principal de optimización
	for iter := 0; iter < maxIter; iter++ {
		cambios := 0

		// Fase A: Asignar cada observación al centroide más cercano
		for i := range datos {
			minDist := math.MaxFloat64
			clusterAsignado := 0

			for j := 0; j < k; j++ {
				dist := calcularDistancia(datos[i].Features, centroides[j])
				if dist < minDist {
					minDist = dist
					clusterAsignado = j
				}
			}

			// Registrar si el punto cambió de grupo
			if datos[i].Cluster != clusterAsignado {
				datos[i].Cluster = clusterAsignado
				cambios++
			}
		}

		// Criterio de parada: Si ningún punto cambió de cluster, hemos terminado
		if cambios == 0 {
			fmt.Printf("-> Convergencia alcanzada en la iteración %d\n", iter)
			break
		}

		// Fase B: Recalcular la posición de los centroides (promedio de sus puntos)
		nuevosCentroides := make([][]float64, k)
		conteos := make([]int, k)
		for i := 0; i < k; i++ {
			nuevosCentroides[i] = make([]float64, numFeatures)
		}

		// Sumar los valores de todas las características por cluster
		for _, obs := range datos {
			c := obs.Cluster
			conteos[c]++
			for j := 0; j < numFeatures; j++ {
				nuevosCentroides[c][j] += obs.Features[j]
			}
		}

		// Dividir por la cantidad de puntos para obtener el nuevo centroide
		for i := 0; i < k; i++ {
			if conteos[i] > 0 {
				for j := 0; j < numFeatures; j++ {
					centroides[i][j] = nuevosCentroides[i][j] / float64(conteos[i])
				}
			}
		}
	}
	return centroides
}

func main() {
	// IMPORTANTE: Ajusta esta ruta según la ubicación real en tu repositorio local
	rutaArchivo := "/content/drive/MyDrive/Concurrente/Procesado/dataset_kmeans_go.csv"

	file, err := os.Open(rutaArchivo)
	if err != nil {
		fmt.Printf("Error abriendo el archivo %s: %v\n", rutaArchivo, err)
		return
	}
	defer file.Close()

	reader := csv.NewReader(file)
	registros, err := reader.ReadAll()
	if err != nil {
		fmt.Println("Error leyendo el CSV:", err)
		return
	}

	var dataset []Observacion

	// Mapeo de columnas: 2:CO, 3:NO2, 4:PM10, 5:PM2.5, 6:SO2
	for i := 1; i < len(registros); i++ {
		valCO, _   := strconv.ParseFloat(registros[i][2], 64)
		valNO2, _  := strconv.ParseFloat(registros[i][3], 64)
		valPM10, _ := strconv.ParseFloat(registros[i][4], 64)
		valPM25, _ := strconv.ParseFloat(registros[i][5], 64)
		valSO2, _  := strconv.ParseFloat(registros[i][6], 64)

		obs := Observacion{
			Features: []float64{valPM10, valPM25, valSO2, valNO2, valCO},
			Cluster:  -1,
		}
		dataset = append(dataset, obs)
	}

	fmt.Printf("Dataset listo. Total de observaciones procesadas: %d\n", len(dataset))

	// Configuración de hiperparámetros
	K := 3           // Cantidad de clusters a encontrar
	MaxIter := 100   // Límite de seguridad para evitar bucles infinitos

	// --- INICIO DE MEDICIÓN DE RENDIMIENTO ---
	inicio := time.Now()

	fmt.Println("Ejecutando algoritmo K-Means Secuencial...")
	centroidesFinales := kMeansSecuencial(dataset, K, MaxIter)

	duracion := time.Since(inicio)
	// --- FIN DE MEDICIÓN ---

	fmt.Printf("\n=== RESULTADOS SECUENCIALES ===\n")
	fmt.Printf("Tiempo de ejecución (T_secuencial): %v\n", duracion)
	fmt.Println("Centroides finales [PM10, PM2.5, SO2, NO2, CO]:")
	for i, c := range centroidesFinales {
		fmt.Printf("  Cluster %d: %.4f\n", i, c)
	}
}

Overwriting main_secuencial.go


In [26]:
!go run main_secuencial.go

Dataset listo. Total de observaciones procesadas: 1035
Ejecutando algoritmo K-Means Secuencial...
-> Convergencia alcanzada en la iteración 15

=== RESULTADOS SECUENCIALES ===
Tiempo de ejecución (T_secuencial): 1.547616ms
Centroides finales [PM10, PM2.5, SO2, NO2, CO]:
  Cluster 0: [18.9841 12.3515 12.7142 6.2845 246.5373]
  Cluster 1: [19.6568 12.5415 14.8296 6.2732 298.2777]
  Cluster 2: [18.3203 11.5569 13.5596 6.5114 358.4277]


### Prueba de Estrés: Ejecución Secuencial Masiva

Como el dataset original de 1 035 datos finalizó en apenas **1.54 ms**, el tiempo es insuficiente para superar el costo de creación de hilos (overhead) de la futura versión concurrente.

Para poder medir el beneficio real de la concurrencia, en esta ejecución sometemos al algoritmo secuencial a una prueba de estrés. Se aplica un **multiplicador (x1000)** en memoria para procesar **1 035 000 observaciones**. El tiempo resultante ($T_{Secuencial}$) servirá como línea base oficial para el cálculo del *Speedup*.

In [21]:
%%writefile main_secuencial_prueba_masiva.go
package main

import (
	"encoding/csv"
	"fmt"
	"math"
	"math/rand"
	"os"
	"strconv"
	"time"
)

// Observacion representa una fila del dataset con sus 5 contaminantes.
type Observacion struct {
	Features []float64 // Vector multivariable: [PM10, PM2.5, SO2, NO2, CO]
	Cluster  int       // Índice del centroide al que pertenece
}

// calcularDistancia aplica la fórmula de distancia euclidiana entre dos vectores.
func calcularDistancia(a, b []float64) float64 {
	sum := 0.0
	for i := range a {
		diff := a[i] - b[i]
		sum += diff * diff
	}
	return math.Sqrt(sum)
}

// kMeansSecuencial ejecuta el algoritmo de agrupamiento en un solo hilo.
func kMeansSecuencial(datos []Observacion, k int, maxIter int) [][]float64 {
	numFeatures := len(datos[0].Features)
	centroides := make([][]float64, k)

	// 1. Inicialización: Seleccionar 'k' centroides iniciales al azar
	rand.Seed(time.Now().UnixNano())
	for i := 0; i < k; i++ {
		idx := rand.Intn(len(datos))
		centroides[i] = make([]float64, numFeatures)
		copy(centroides[i], datos[idx].Features)
	}

	// 2. Bucle principal de optimización
	for iter := 0; iter < maxIter; iter++ {
		cambios := 0

		// Fase A: Asignar cada observación al centroide más cercano
		for i := range datos {
			minDist := math.MaxFloat64
			clusterAsignado := 0

			for j := 0; j < k; j++ {
				dist := calcularDistancia(datos[i].Features, centroides[j])
				if dist < minDist {
					minDist = dist
					clusterAsignado = j
				}
			}

			// Registrar si el punto cambió de grupo
			if datos[i].Cluster != clusterAsignado {
				datos[i].Cluster = clusterAsignado
				cambios++
			}
		}

		// Criterio de parada: Si ningún punto cambió de cluster, hemos terminado
		if cambios == 0 {
			fmt.Printf("-> Convergencia alcanzada en la iteración %d\n", iter)
			break
		}

		// Fase B: Recalcular la posición de los centroides (promedio de sus puntos)
		nuevosCentroides := make([][]float64, k)
		conteos := make([]int, k)
		for i := 0; i < k; i++ {
			nuevosCentroides[i] = make([]float64, numFeatures)
		}

		// Sumar los valores de todas las características por cluster
		for _, obs := range datos {
			c := obs.Cluster
			conteos[c]++
			for j := 0; j < numFeatures; j++ {
				nuevosCentroides[c][j] += obs.Features[j]
			}
		}

		// Dividir por la cantidad de puntos para obtener el nuevo centroide
		for i := 0; i < k; i++ {
			if conteos[i] > 0 {
				for j := 0; j < numFeatures; j++ {
					centroides[i][j] = nuevosCentroides[i][j] / float64(conteos[i])
				}
			}
		}
	}
	return centroides
}

func main() {
	rutaArchivo := "/content/drive/MyDrive/Concurrente/Procesado/dataset_kmeans_go.csv"

	file, err := os.Open(rutaArchivo)
	if err != nil {
		fmt.Printf("Error abriendo el archivo %s: %v\n", rutaArchivo, err)
		return
	}
	defer file.Close()

	reader := csv.NewReader(file)
	registros, err := reader.ReadAll()
	if err != nil {
		fmt.Println("Error leyendo el CSV:", err)
		return
	}

	var datasetBase []Observacion

	// Mapeo de columnas: 2:CO, 3:NO2, 4:PM10, 5:PM2.5, 6:SO2
	for i := 1; i < len(registros); i++ {
		valCO, _   := strconv.ParseFloat(registros[i][2], 64)
		valNO2, _  := strconv.ParseFloat(registros[i][3], 64)
		valPM10, _ := strconv.ParseFloat(registros[i][4], 64)
		valPM25, _ := strconv.ParseFloat(registros[i][5], 64)
		valSO2, _  := strconv.ParseFloat(registros[i][6], 64)

		obs := Observacion{
			Features: []float64{valPM10, valPM25, valSO2, valNO2, valCO},
			Cluster:  -1,
		}
		datasetBase = append(datasetBase, obs)
	}

	// =========================================================
	// PRUEBA DE ESTRÉS: Multiplicador x1000
	// Generamos la misma carga que en la versión concurrente
	// =========================================================
	var datasetMasivo []Observacion
	multiplicador := 1000
	for i := 0; i < multiplicador; i++ {
		datasetMasivo = append(datasetMasivo, datasetBase...)
	}

	fmt.Printf("Total de observaciones masivas procesadas: %d\n", len(datasetMasivo))

	// Configuración de hiperparámetros
	K := 3           // Cantidad de clusters a encontrar
	MaxIter := 100   // Límite de seguridad para evitar bucles infinitos

	// --- INICIO DE MEDICIÓN DE RENDIMIENTO ---
	inicio := time.Now()

	fmt.Println("Ejecutando algoritmo K-Means Secuencial Masivo...")
	centroidesFinales := kMeansSecuencial(datasetMasivo, K, MaxIter)

	duracion := time.Since(inicio)
	// --- FIN DE MEDICIÓN ---

	fmt.Printf("\n=== RESULTADOS SECUENCIALES ===\n")
	fmt.Printf("Tiempo de ejecución (T_secuencial): %v\n", duracion)
	fmt.Println("Centroides finales [PM10, PM2.5, SO2, NO2, CO]:")
	for i, c := range centroidesFinales {
		fmt.Printf("  Cluster %d: %.4f\n", i, c)
	}
}

Writing main_secuencial_prueba_masiva.go


In [31]:
!go run main_secuencial_prueba_masiva.go

Total de observaciones masivas procesadas: 1035000
Ejecutando algoritmo K-Means Secuencial Masivo...
-> Convergencia alcanzada en la iteración 17

=== RESULTADOS SECUENCIALES ===
Tiempo de ejecución (T_secuencial): 969.830808ms
Centroides finales [PM10, PM2.5, SO2, NO2, CO]:
  Cluster 0: [18.9841 12.3515 12.7142 6.2845 246.5373]
  Cluster 1: [19.6568 12.5415 14.8296 6.2732 298.2777]
  Cluster 2: [18.3203 11.5569 13.5596 6.5114 358.4277]
